In [ ]:
# scarico i dati da yfinance
import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession


import xlrd

In [6]:
# verify Java installation for PySpark
import os, sys

# print Java version if available
try:
    java_version = os.popen('java -version 2>&1').read()
    print("Java version information:\n", java_version)
except Exception as e:
    print(f"Error checking Java version: {e}")

# if needed, set JAVA_HOME manually
# os.environ['JAVA_HOME'] = r"C:\Program Files\Java\jdk-XX"



Java version information:
 'java' is not recognized as an internal or external command,
operable program or batch file.



In [ ]:
# list of tickers needed for the analysis

equity_tickers = ['^GSPC', 'FEZ']
energy_tickers = [ 'CL=F', 'NG=F']
fx_tickers = ['EURUSD=X', 'USDCHF=X']
vix_ticker = '^VIX'
all_tickers = equity_tickers + energy_tickers + fx_tickers + [vix_ticker]

# ^GSPC = S&P500
# FEZ = stoxx50
# CL=F Crude oil WTI, BZ=F Crude oil Brent, NG=F Natural gas

start_date = "2004-01-05"
end_date = "2025-12-31"
df_list = []

# Creiamo l'oggetto Ticker
for el in all_tickers:
    
    df = yf.download(el, start=start_date, end=end_date)['Close']
    # print(df.shape)
    df.name= el
    df_list.append(df)

df_yfinance_data = pd.concat(df_list, axis = 1)



# prezzi giornalieri per il Brent Crude Oil dal 1987
brent_eia = pd.read_excel("https://www.eia.gov/dnav/pet/hist_xls/RBRTEd.xls", sheet_name="Data 1", skiprows=2)

brent_eia['Date'] = pd.to_datetime(brent_eia['Date'])
brent_eia.set_index('Date', inplace=True)

brent_eia = brent_eia.loc[start_date:]
brent_eia.columns = ['Brent_EIA']


df_raw = pd.concat([df_yfinance_data, brent_eia], axis=1)
print(df_raw.head())


In [36]:
print(df_raw.isna().sum(), df_raw.shape)

^GSPC        219
FEZ          219
CL=F         222
NG=F         221
EURUSD=X      48
USDCHF=X      35
^VIX         219
Brent_EIA    160
dtype: int64 (5752, 8)


In [37]:

df_raw_no_na = df_raw.dropna()
print(df_raw_no_na.shape)


(5446, 8)


## Read csv file in pyspark

In [4]:
import os
# os.environ['JAVA_HOME'] = r"C:\Java\jdk-11.0.18"
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataRetrieval").getOrCreate()
print(spark)

file_path = fr"C:\Users\mdenova\OneDrive - KPMG\Desktop\Varie\corso python\df_historical_prices.csv"
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
df_spark.display()

UnsupportedOperationException: getSubject is not supported

In [ ]:
file_path = fr"C:\Users\mdenova\OneDrive - KPMG\Desktop\Varie\corso python\df_historical_prices.csv"
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
df_spark.display()

## Feature engineering

In [10]:

import yaml
from box import Box
import os
import pandas as pd
from config_path import cfg_path
import numpy as np



df_raw = pd.read_csv(os.path.join(cfg_path.workspace_root, cfg_path.raw_data))
print(df_raw.head())



         Date        ^GSPC        FEZ       CL=F   NG=F  EURUSD=X  USDCHF=X  \
0  2004-01-05  1122.219971  17.796146  33.779999  6.827  1.268698    1.2320   
1  2004-01-06  1123.670044  17.890413  33.700001  7.082  1.272103    1.2319   
2  2004-01-07  1126.329956  17.642353  33.619999  6.878  1.264095    1.2395   
3  2004-01-08  1131.920044  17.994598  33.980000  7.094  1.277498    1.2266   
4  2004-01-09  1121.859985  17.835844  34.310001  7.287  1.285892    1.2212   

    ^VIX  Brent_EIA  
0  17.49      32.30  
1  16.73      31.20  
2  15.50      30.99  
3  15.61      31.11  
4  16.75      31.91  


In [15]:
# compute log returns for each asset in the dataframe

log_returns = np.log(df_raw.iloc[:, 1:] / df_raw.iloc[:, 1:].shift(1))
print(log_returns.head())
# df_raw['log_returns'] = np.log(df_raw['^GSPC'] / df_raw['^GSPC'].shift(1))

      ^GSPC       FEZ      CL=F      NG=F  EURUSD=X  USDCHF=X      ^VIX  \
0       NaN       NaN       NaN       NaN       NaN       NaN       NaN   
1  0.001291  0.005283 -0.002371  0.036671  0.002681 -0.000081 -0.044426   
2  0.002364 -0.013963 -0.002377 -0.029228 -0.006315  0.006150 -0.076363   
3  0.004951  0.019769  0.010651  0.030921  0.010547 -0.010462  0.007072   
4 -0.008927 -0.008861  0.009665  0.026843  0.006549 -0.004412  0.070487   

   Brent_EIA  
0        NaN  
1  -0.034649  
2  -0.006754  
3   0.003865  
4   0.025390  


c:\Users\mdenova\AppData\Local\miniforge3\envs\env_test\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [ ]:


# percentage return
returns = df.pct_change()

# 1-month and 3-months rolling volatility
volatility_1m = log_returns.rolling(window=21).std()
volatility_3m = log_returns.rolling(window=63).std()

# rolling momentum
momentum_1m = log_returns.rolling(window=21).mean()
momentum_3m = log_returns.rolling(window=63).mean()

# correlation between assets
correlation_matrix = log_returns.corr()

#  computation of rolling correlation between S&P500 and VIX
rolling_corr_spx_vix = (
    log_returns['^GSPC']
    .rolling(63)
    .corr(log_returns['^VIX'])
)

# realized volatility
realized_vol = log_returns.rolling(21).std() * np.sqrt(252)

# drawdowns
cum_returns = (1 + returns).cumprod()
rolling_max = cum_returns.cummax()
drawdown = cum_returns / rolling_max - 1




